In [1]:
import os                                     
import tensorflow as tf
from tensorflow.keras import layers, models          # type: ignore 
from tensorflow.keras.layers import Dropout          # type: ignore 
from tensorflow.keras.preprocessing.image import ImageDataGenerator          # type: ignore 
from tensorflow.keras.callbacks import EarlyStopping                         # type: ignore 

# SECTION 1 — TRAIN DATA AUGMENTATION : “image transformation machine” ----->  image processing pipeline ban rahi hai.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2
    )

val_datagen = ImageDataGenerator(rescale=1./255)

# SECTION 3: TRAIN GENERATOR
train_generator = train_datagen.flow_from_directory(
    'C:/Users/sanch/Documents/Git_hub/Streamlit-KrishiNetra/02_dataset/01_train',
    target_size=(150, 150),
    batch_size=8,
    class_mode='categorical'
)

# SECTION 4: VALIDATION GENERATOR
val_generator = val_datagen.flow_from_directory(
    'C:/Users/sanch/Documents/Git_hub/Streamlit-KrishiNetra/02_dataset/02_validate/',
    target_size=(150, 150),
    batch_size=8,
    class_mode='categorical'
)


# 3. Build CNN Model 2 :

model = models.Sequential()

#-----------------------------------------------------------------------------------------------------------------------#

# C1: Convolution
model.add(layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same', input_shape=(150,150,3)))  #padding='same' ----> preserve edge information

# C2: Convolution
model.add(layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same'))  #padding='same' ----> preserve edge information


# P1: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))


#-----------------------------------------------------------------------------------------------------------------------#

# C3: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))  #padding='same' ----> preserve edge information

# C4: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))

# P2: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#-----------------------------------------------------------------------------------------------------------------------#

# C5: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))


# C6: Convolution
model.add(layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same'))

# P3: Max Pooling
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#-----------------------------------------------------------------------------------------------------------------------#

# Flatten
model.add(layers.GlobalAveragePooling2D())    # GlobalAveragePooling ----->reduce memorization

#-----------------------------------------------------------------------------------------------------------------------#

# Fully Connected Layers
model.add(layers.Dense(128, activation='relu'))
model.add(Dropout(0.3))                            # stronger dropout---------->better generalization
model.add(layers.Dense(64, activation='relu'))
model.add(Dropout(0.3))

#-----------------------------------------------------------------------------------------------------------------------#

# Output Layer
model.add(layers.Dense(19, activation='softmax'))




# 4. Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])



early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)

Found 18085 images belonging to 19 classes.
Found 4515 images belonging to 19 classes.


c:\Users\sanch\Documents\Git_hub\Streamlit-KrishiNetra\krishi_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [2]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 150, 150, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 75, 75, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 37, 37, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 37, 37, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 19)             │         1,235 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157,235 (614.20 KB)

 Trainable params: 157,235 (614.20 KB)

 Non-trainable params: 0 (0.00 B)